# Import Libraries

In [5]:
from pathlib import Path
import pandas as pd
import duckdb

# Global Variables

In [6]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_PATH = PROCESSED_DIR / "data_quality_report.md"

# Import raw data

In [19]:
excel_sheets = pd.read_excel(
    RAW_DATA_PATH,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    engine="openpyxl",
)

raw = pd.concat(excel_sheets.values(), ignore_index=True)

raw.columns = (
    raw.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '')
)

raw.head()

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# Processing

## Missing Values

In [27]:
df = raw.copy()

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1067371 non-null  object        
 1   stockcode    1067371 non-null  object        
 2   description  1062989 non-null  object        
 3   quantity     1067371 non-null  int64         
 4   invoicedate  1067371 non-null  datetime64[us]
 5   price        1067371 non-null  float64       
 6   customerid   824364 non-null   float64       
 7   country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [29]:
df.describe()

,quantity,invoicedate,price,customerid
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [30]:
missing = df.isna().sum().to_frame("missing_cnt")
missing["missing_pct"] = (missing["missing_cnt"] / len(raw) * 100).round(2)
missing

,missing_cnt,missing_pct
invoice,0,0.00
stockcode,0,0.00
description,4382,0.41
quantity,0,0.00
invoicedate,0,0.00
price,0,0.00
customerid,243007,22.77
country,0,0.00


## Duplicates

In [41]:
duplicate_rows = df.duplicated(keep=False)

print(f"Rows involved in duplication: {duplicate_rows.sum():,} ({duplicate_rows.sum()/len(df):.1%})")

display(df.loc[duplicate_rows].sort_values(["invoice", "stockcode"]).head(20))

Rows involved in duplication: 67,242 (6.3%)


,invoice,stockcode,description,quantity,invoicedate,price,customerid,country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


## Enforce data types

In [31]:
df["customerid"] = df["customerid"].astype("string")
df["invoice"] = df["invoice"].astype("string")
df["stockcode"] = df["stockcode"].astype("string")
df["country"] = df["country"].astype("string")

df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

print(f"Date range: {df['invoicedate'].min()} to {df['invoicedate'].max()}")
print(f"Invalid dates: {df['invoicedate'].isna().sum():,}")
print(f"Invalid quantities: {df['quantity'].isna().sum():,}")
print(f"Invalid prices: {df['price'].isna().sum():,}")

df[["invoicedate", "quantity", "price"]].describe()

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Invalid dates: 0
Invalid quantities: 0
Invalid prices: 0


,invoicedate,quantity,price
count,1067371,1.067371e+06,1.067371e+06
mean,2011-01-02 21:13:55.394029,9.938898e+00,4.649388e+00
min,2009-12-01 07:45:00,-8.099500e+04,-5.359436e+04
25%,2010-07-09 09:46:00,1.000000e+00,1.250000e+00
50%,2010-12-07 15:28:00,3.000000e+00,2.100000e+00
75%,2011-07-22 10:23:00,1.000000e+01,4.150000e+00
max,2011-12-09 12:50:00,8.099500e+04,3.897000e+04
std,NaN,1.727058e+02,1.235531e+02
